In [ ]:
!wget "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import random

load_dotenv()

# 로컬
ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

# 클라우드
# from google.colab import userdata
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# os.environ["WANDB_PROJECT"] = "patent_disc"
# os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
# os.environ["HF_HOME"] = ".hf_cache"
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import load_dataset
from sklearn.metrics import f1_score

In [3]:
# Config
config = {
    "num_labels": 188,
    "seed": 42,
    "learning_rate": 3e-5,
    "epochs": 12,
    "early_stop": 6,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "model_name": "skt/A.X-Encoder-base",
    "max_len": 8192,          # >8,192 극소수(<1%)만 절단 → 사실상 full (코퍼스 max 10,523)
    "eff_batch": 8,           # max_len=512 런과 유효 배치 등화 → 컨텍스트 길이 외 변수 고정
    "micro_batch": 2,         
    "eval_micro_batch": 4,   
    "repo_train": "ingyoun/A.X-patent-maxlen8192-train",
    "repo_final": "ingyoun/A.X-patent-maxlen8192",
    "run_name": "modernbert_maxlen=8192_v2",
    "out_path": "/content/output/modernbert-maxlen8192",
    "tag": "modernbert-patent-len8192",
}
config["grad_accum"] = config["eff_batch"] // config["micro_batch"]   # micro×accum = eff_batch

In [4]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [5]:
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

cpu


## 데이터셋

In [ ]:
dataset = load_dataset("ingyoun/patent-clean-text-modernbert-tokenized")
dataset   # 사전토큰화본 — truncation 없이 저장됨(최대 10,523 토큰). 절단은 토크나이저 로드 후 수행.

## 토크나이저

In [ ]:
REV = "9708f9c404ace91efd25c06fac2d73413616f4ef"
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], revision=REV)
EOS_ID = tokenizer.eos_token_id


def _prep(batch):
    max_len = config["max_len"]
    ids, masks = [], []
    for x, m in zip(batch["input_ids"], batch["attention_mask"]):
        if len(x) > max_len:
            x = x[: max_len - 1] + [EOS_ID]   # <s> 유지 + 꼬리를 <\s>로 마감
            m = m[:max_len]
        ids.append(x)
        masks.append(m)
    return {"input_ids": ids, "attention_mask": masks, "length": [len(i) for i in ids]}


dataset = dataset.map(_prep, batched=True)
print(f"EOS_ID={EOS_ID}  max_len={config['max_len']}")
dataset

## 모델

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=config["model_name"],
        num_labels=config["num_labels"], 
        problem_type="multi_label_classification", 
        classifier_dropout=0.5,
        dtype=torch.float32,
        attn_implementation="flash_attention_2",   # 장문 attention 메모리·속도. autocast(fp16/bf16)가 q/k/v를 반정밀로 공급
    )   

## 커스텀

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: int = 2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

In [ ]:
class FocalTrainer(Trainer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.focal = FocalLoss(0.25, 2.0)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        loss = self.focal(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

In [ ]:
class MultiLabelCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer
    
    def __call__(self, feats):
        labels = torch.tensor([f["labels"] for f in feats], dtype=torch.float)
        keys = ("input_ids", "attention_mask")
        enc = [{k: f[k] for k in keys if k in f} for f in feats]
        batch = self.tok.pad(enc, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch

In [ ]:
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(eval_pred):
    """03_02_Metric의 headline과 동일 정의 — sigmoid→τ=0.5 멀티라벨 F1(micro/macro/sample).
    micro_f1이 모델 선택(metric_for_best_model) 기준. 벤더 연속성 앵커(top-1 weighted)도 병기.
    """
    logits, labels = eval_pred
    logits = np.asarray(logits)
    Y = np.asarray(labels).astype(int)
    pred = (_sigmoid(logits) >= 0.5).astype(int)
    return {
        "micro_f1":  f1_score(Y, pred, average="micro",   zero_division=0),   # headline (모델 선택 기준)
        "macro_f1":  f1_score(Y, pred, average="macro",   zero_division=0),
        "sample_f1": f1_score(Y, pred, average="samples", zero_division=0),
        "empty_rate": float((pred.sum(1) == 0).mean()),
        "anchor_weighted_f1": f1_score(Y.argmax(1), logits.argmax(1), average="weighted", zero_division=0),  # 벤더 연속성
    }

## 훈련

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/results',
    seed=config["seed"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    lr_scheduler_type="linear",
    warmup_ratio=config["warmup_ratio"],
    per_device_train_batch_size=config["micro_batch"],
    per_device_eval_batch_size=config["eval_micro_batch"],
    gradient_accumulation_steps=config["grad_accum"],   # micro×accum = eff_batch (max_len=512 런과 등화)
    train_sampling_strategy="group_by_length",          # 유사 길이 배치로 padding 최소화 (v5에서 group_by_length 대체)
    remove_unused_columns=False,                        # 커스텀 collator가 키를 직접 선택 + length 컬럼 보존
    num_train_epochs=config["epochs"],
    bf16=True,                                          # ModernBERT 계열 안정성엔 bf16 (bf16=True, fp16=False)
    eval_strategy='steps',
    eval_steps=12618,
    save_strategy='steps',
    save_steps=12618,
    save_total_limit=3,
    logging_dir='/content/logs',
    logging_steps=50,
    metric_for_best_model="micro_f1",                   # 03_02 headline 기준 모델 선택
    greater_is_better=True,
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id=config["repo_train"],
    hub_strategy="checkpoint",
    report_to="wandb",
    run_name=config["run_name"]
)

In [ ]:
trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=MultiLabelCollator(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stop"])]
)

In [ ]:
## micro batch 확인 — worst-case 배치로 실제 스텝을 돌려 peak 측정
def probe_batches(model, vocab_size, seq_len, train_mb=(1, 2, 4), eval_mb=(2, 4, 8), num_labels=188):
    """worst case = 모든 시퀀스가 seq_len이고 mask가 전부 1(패딩 0).

    group_by_length가 all-long 배치를 반드시 만들므로 이 경우는 확실히 발생한다
    (random 샘플링이어도 배치에 장문 하나만 섞이면 배치 전체가 그 길이로 패딩된다).

    - AdamW state(exp_avg/exp_avg_sq ≈ params×8B)를 상주시킨 **정상상태** peak를 잰다.
      lr=0이라 가중치는 변하지 않는다 → 사전학습 가중치를 훼손하지 않음.
    - eval도 잰다: forward-only(no_grad)라 싸지만 eval 배치 역시 배치 내 최댓값으로 패딩되고,
      eval OOM은 첫 에폭 끝에서 런을 죽인다.
    """
    assert model.device.type == "cuda", "모델이 GPU에 없습니다. — Trainer 생성 후 실행할 것"
    dev = model.device
    opt = torch.optim.AdamW(model.parameters(), lr=0.0)   # lr=0 → state만 할당, 가중치 불변

    def _mk(mb):
        ids = torch.randint(5, vocab_size, (mb, seq_len), device=dev)
        return ids, torch.ones_like(ids), torch.zeros(mb, num_labels, device=dev)

    model.train()
    for mb in train_mb:
        try:
            ids, mask, labels = _mk(mb)
            for step in (0, 1):                       # step0: AdamW state 할당 / step1: 정상상태 peak 측정
                if step == 1:
                    torch.cuda.reset_peak_memory_stats()
                with torch.autocast("cuda", dtype=torch.bfloat16):   # Trainer(bf16=True)와 동일 경로
                    out = model(input_ids=ids, attention_mask=mask)
                    loss = F.binary_cross_entropy_with_logits(out.logits.float(), labels)
                loss.backward()                                      # backward는 autocast 밖
                opt.step()
                opt.zero_grad(set_to_none=True)
            print(f"train micro={mb:>2}: peak {torch.cuda.max_memory_allocated()/1e9:5.1f} GB  OK")
        except torch.cuda.OutOfMemoryError:
            print(f"train micro={mb:>2}: OOM")
            opt.zero_grad(set_to_none=True)
            break                                     # 이후 후보는 자명하게 OOM
        finally:
            torch.cuda.empty_cache()

    model.eval()
    for mb in eval_mb:                                # 옵티마이저 state가 상주한 실제 조건에서 측정
        try:
            torch.cuda.reset_peak_memory_stats()
            ids, mask, _ = _mk(mb)
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                model(input_ids=ids, attention_mask=mask)
            print(f"eval  micro={mb:>2}: peak {torch.cuda.max_memory_allocated()/1e9:5.1f} GB  OK")
        except torch.cuda.OutOfMemoryError:
            print(f"eval  micro={mb:>2}: OOM")
            break
        finally:
            torch.cuda.empty_cache()

    del opt                                           # Trainer가 자체 옵티마이저를 새로 만들도록 정리
    model.zero_grad(set_to_none=True)
    model.train()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()              # train() peak를 깨끗하게 재측정하기 위함


# 설정값 검증. 여유를 보려면 train_mb=(1, 2, 4), eval_mb=(2, 4, 8)로 넓혀 실행
probe_batches(
    model, tokenizer.vocab_size, config["max_len"],
    train_mb=(config["micro_batch"],),
    eval_mb=(config["eval_micro_batch"],),
    num_labels=config["num_labels"],
)

In [ ]:
import gc

# probe_batches가 옵티마이저·캐시를 내부에서 정리하므로 여기선 GC만 한 번 더 돌린다
gc.collect()
torch.cuda.empty_cache()
print(f"probe 후 잔여 allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB (모델 가중치)")

In [ ]:
trainer.train()

## 평가

In [ ]:
test_metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

In [ ]:
os.makedirs(config["out_path"], exist_ok=True)
trainer.save_model(config["out_path"])
tokenizer.save_pretrained(config["out_path"])

metrics_fp = os.path.join(config["out_path"], f"{config['tag']}_test_metrics.json")
with open(metrics_fp, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print("saved", config["out_path"])  

In [ ]:
trainer.push_to_hub(config["repo_final"])
tokenizer.push_to_hub(config["repo_final"])